In [155]:
import sys
sys.path.append('../') 
from visualisation import *
import xarray as xr
import dask
import geopandas as gpd
from shapely.geometry import Point
from geopy.geocoders import Nominatim
from tqdm import tqdm  # For progress bar in large datasets
from geopy.exc import GeocoderTimedOut

In [52]:
import time
from geopy.exc import GeocoderTimedOut
from geopy.geocoders import Nominatim

def geocode_with_retry(address, retries=3):
    geolocator = Nominatim(user_agent="geoapi", timeout=30)  # Increase timeout
    for attempt in range(retries):
        try:
            location = geolocator.geocode(address)
            return location
        except GeocoderTimedOut:
            print(f"Geocoding timed out. Attempt {attempt + 1}/{retries}. Retrying...")
            time.sleep(2)  # Wait before retrying
    print("Failed to geocode after several attempts.")
    return None

address = "1600 Pennsylvania Ave NW, Washington, DC"
location = geocode_with_retry(address)
if location:
    print(location)

White House, 1600, Pennsylvania Avenue Northwest, Ward 2, Washington, District of Columbia, 20500, United States


In [4]:
bom_path = "/mnt/d/bom_nci/2023/01/01/"
files = glob(bom_path+"*.nc")
len(files)

103

In [16]:
df = [xr.open_dataset(file).to_dataframe() for file in files[:15]]
df = pd.concat(df, axis=0).reset_index(drop=False)
df = df.dropna(subset='direct_normal_irradiance').reset_index(drop=True)
df['julian_date'] = pd.to_datetime(df['julian_date'], origin='julian', unit='D')


In [22]:
df0 = df.groupby(['latitude',	'longitude']).agg({'julian_date': lambda x:x.unique(),'time': lambda x:x.unique()}).reset_index(drop=False)

In [30]:
df.head()

,time,latitude,longitude,crs,surface_global_irradiance,direct_normal_irradiance,surface_diffuse_irradiance,quality_mask,cloud_type,cloud_optical_depth,solar_elevation,solar_azimuth,julian_date,geometry
0,2022-12-31 18:30:00,-44.5,145.919998,-2147483647,0.0,0.0,0.0,4.0,0.0,0.0,-0.3,123.6,2022-12-31 18:38:34.534297582,POINT (145.9199981689453 -44.5)
1,2022-12-31 18:30:00,-44.5,145.940002,-2147483647,0.0,0.0,0.0,4.0,0.0,0.0,-0.3,123.6,2022-12-31 18:38:34.534297582,POINT (145.94000244140625 -44.5)
2,2022-12-31 18:30:00,-44.5,145.960007,-2147483647,0.0,0.0,0.0,4.0,0.0,0.0,-0.3,123.6,2022-12-31 18:38:34.534297582,POINT (145.9600067138672 -44.5)
3,2022-12-31 18:30:00,-44.5,145.979996,-2147483647,0.0,0.0,0.0,4.0,0.0,0.0,-0.2,123.6,2022-12-31 18:38:34.534297582,POINT (145.97999572753906 -44.5)
4,2022-12-31 18:30:00,-44.5,146.000000,-2147483647,0.0,0.0,0.0,4.0,0.0,0.0,-0.2,123.6,2022-12-31 18:38:34.534297582,POINT (146 -44.5)


In [ ]:
geolocator = Nominatim(user_agent="geoapi")
# Convert to GeoDataFrame
df['geometry'] = [Point(x,y) for x,y in zip(df['longitude'], df['latitude'])]


In [ ]:
points = df['geometry'].unique()

array([<POINT (145.92 -44.5)>, <POINT (145.94 -44.5)>,
       <POINT (145.96 -44.5)>, ..., <POINT (126.18 -10)>,
       <POINT (126.2 -10)>, <POINT (126.22 -10)>],
      shape=(2146637,), dtype=object)

In [48]:
df3 = df.drop_duplicates(subset='geometry').reset_index(drop=True)

In [53]:
df3['address'] = df3['geometry'].apply(lambda x: gpd.tools.reverse_geocode(x))


GeocoderTimedOut: Service timed out

In [70]:
geo_list = df['geometry'].to_list()
i = 0

In [163]:
i=0

In [ ]:
address['address'].to_list()[0]==None

True

In [164]:
i

0

In [133]:
if address['address'].to_list()[0]==None:
    print('hey')

hey


In [165]:
address_i = []
i_i = []
for _ in range(len(geo_list)):
    try:
        address = gpd.tools.reverse_geocode(geo_list[i], timeout=60)
        if address['address'].to_list()[0]!=None:
            print(address['address'], i)
            address_i.append(address)
            i_i.append(i)
    except Exception as e:
        pass
        print(e, i)
    i += 1


0    Huon Marine Park
Name: address, dtype: object 1313
0    Pedra Branca, Huon Valley, Tasmania, Australia
Name: address, dtype: object 4505
0    Eddystone, Huon Valley, Tasmania, Australia
Name: address, dtype: object 4506
0    Tasman Fracture Marine National Park Zone, Aus...
Name: address, dtype: object 4649
0    Sidmouth Rock, Tasmania, Australia
Name: address, dtype: object 4684
0    Mewstone, Huon Valley, Tasmania, Australia
Name: address, dtype: object 5540
0    Needles Rocks, Tasmania, Australia
Name: address, dtype: object 6240
0    Needles Rocks, Huon Valley, Tasmania, Australia
Name: address, dtype: object 6241
0    Keepers Track, Huon Valley, Tasmania, Australia
Name: address, dtype: object 6242
0    Mattsuyker Island, Huon Valley, Tasmania, Aust...
Name: address, dtype: object 6418
0    Flat Top Island, Huon Valley, Tasmania, Australia
Name: address, dtype: object 6423
0    South Cape, Huon Valley, Tasmania, Australia
Name: address, dtype: object 6439
0    South East Cape

KeyboardInterrupt: 

In [ ]:

# Add a reverse geocoding function
def get_postcode(point):
    location = geolocator.reverse((point.y, point.x), exactly_one=True)
    if location and 'postcode' in location.raw['address']:
        return location.raw['address']['postcode']
    return None

# Apply with progress bar
tqdm.pandas()
gdf['Postcode'] = gdf['geometry'].progress_apply(get_postcode)

print(gdf)

In [25]:
np.sort(df0.loc[0, 'julian_date'])

array(['2020-11-19T18:38:35.534292158', '2020-11-19T18:48:35.534296630',
       '2020-11-19T18:58:35.534301101', '2020-11-19T19:08:35.534305573',
       '2020-11-19T19:18:35.534310045', '2020-11-19T19:28:35.534274280',
       '2020-11-19T19:38:35.534278752', '2020-11-19T19:48:35.534283223',
       '2020-11-19T19:58:35.534287695'], dtype='datetime64[ns]')

In [27]:
np.sort(df0.loc[10, 'julian_date'])

array(['2020-11-19T18:38:35.534292158', '2020-11-19T18:48:35.534296630',
       '2020-11-19T18:58:35.534301101', '2020-11-19T19:08:35.534305573',
       '2020-11-19T19:18:35.534310045', '2020-11-19T19:28:35.534274280',
       '2020-11-19T19:38:35.534278752', '2020-11-19T19:48:35.534283223',
       '2020-11-19T19:58:35.534287695'], dtype='datetime64[ns]')

In [28]:
np.sort(df0.loc[22, 'time'])

array(['2020-11-19T18:30:00.000000000', '2020-11-19T18:40:00.000000000',
       '2020-11-19T18:50:00.000000000', '2020-11-19T19:00:00.000000000',
       '2020-11-19T19:10:00.000000000', '2020-11-19T19:20:00.000000000',
       '2020-11-19T19:30:00.000000000', '2020-11-19T19:40:00.000000000',
       '2020-11-19T19:50:00.000000000'], dtype='datetime64[ns]')

In [ ]:

# print(df.columns)
# print(df['julian_date'][0])
df = df.query('direct_normal_irradiance > 0').drop_duplicates(subset='julian_date').loc[:, ['julian_date', 'direct_normal_irradiance', 'surface_global_irradiance',
 'surface_diffuse_irradiance', 'cloud_optical_depth', 'cloud_type']]
print(df['julian_date'].min(), df['julian_date'].max())
print(df[:30])